# SEM SimCLR / BYOL — Colab pipeline

Обучение contrastive моделей на SEM-тайлах с полным resume-safe чекпоинтом,
TensorBoard-логированием в Drive, unified evaluation.

**Подготовка:** `Среда выполнения` → `Сменить среду выполнения` → T4 / A100 / V100.

**Синхронизация с кодовой базой:**
- Checkpoint format: полный dict (model/optimizer/scheduler/val_loss) из PR #2.
- Projection head: v2 (с BatchNorm), унифицировано с Crystal (PR #3).
- BYOL tau: cosine ramp-up `base → 1.0` (PR #2).
- Seeds: зафиксированы глобально через `src.utils.repro.set_global_seed(42)` (PR #1).
- Augmentations: параметризованы через CLI (`--crop_scale_min`, `--blur_p`, `--noise_std` и т.д., PR #8).
- Validation split: опциональный (`--val_frac 0.1`), best-checkpoint по val_loss (PR #7).
- TensorBoard: inline-viewer в Colab (PR #7).
- Unified evaluation: `run_sem_evaluation.py` выдаёт markdown-отчёт с bootstrap CI (PR #7).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# === Код из GitHub + данные из Drive + зависимости ===
import os

# 1. Клонируем / обновляем код
if os.path.exists('/content/diploma_project/.git'):
    !cd /content/diploma_project && git pull
else:
    !git clone https://github.com/daniilctrl/sem-image-analysis.git /content/diploma_project

# 2. Распаковываем данные из zip (только если ещё не распакованы)
if not os.path.exists('/content/diploma_project/data/processed/tiles_metadata.csv'):
    !cp "/content/drive/MyDrive/diploma_data/processed_data_v2.zip" /content/
    !unzip -qo /content/processed_data_v2.zip -d /content/_data_tmp/

    # zip хранит файлы внутри processed/, поэтому копируем содержимое,
    # а не саму папку, чтобы не получить data/processed/processed/
    !mkdir -p /content/diploma_project/data/processed
    !cp -r /content/_data_tmp/processed/* /content/diploma_project/data/processed/
    !rm -rf /content/_data_tmp /content/processed_data_v2.zip
    n = len(os.listdir('/content/diploma_project/data/processed'))
    print(f"Data extracted: {n} files")
    assert n > 100, f"Expected ~23904 files, got {n} — check zip contents"
else:
    print('Data already in place.')

# 3. Установка проекта в editable режиме с extras (viz + dev для tensorboard)
%cd /content/diploma_project
!pip install -q -e ".[viz,dev]"

# Быстрая проверка: pyproject работает, smoke-тесты парсятся.
!python -c "from src.utils.repro import set_global_seed; set_global_seed(42); print('repro OK')"
!python -c "from src.models.deep_clustering.augmentations import AugmentationConfig; print('aug config:', AugmentationConfig().to_dict())"

---
## 3. Обучение

Два режима:

- **Один run вручную** (ячейки 3a–4): запусти одну из ячеек с нужной конфигурацией.
- **Sweep серией** (3e): запустит несколько runs последовательно по YAML-конфигу.

Ячейки:

- **3a** — SimCLR с нуля, default aug, `τ=0.5`, `val_frac=0.1`
- **3b** — SimCLR resume после обрыва сессии Colab
- **3c** — SimCLR alternative (`τ=0.2` + strong crop, для ablation)
- **4**  — BYOL с cosine tau ramp-up
- **3d** — TensorBoard live-viewer (запускать параллельно с обучением или sweep)
- **3e** — Sweep: серия экспериментов из `configs/sweeps/sem_default.yaml`

In [ ]:
# === 3a: SimCLR — первый запуск (full resume-safe pipeline) ===
# Полный формат чекпоинта: сохраняются model + optimizer + scheduler + val_loss.
# TensorBoard-логи на Drive (можно открыть ниже в ячейке 3d).
# Val split 10% — best-checkpoint будет выбран по val_loss, не train avg.

RUN_NAME = "simclr_v2_t05_default_aug"

!python /content/diploma_project/src/models/deep_clustering/train.py \
    --data_dir /content/diploma_project/data/processed \
    --metadata_path /content/diploma_project/data/processed/tiles_metadata.csv \
    --output_dir /content/drive/MyDrive/diploma_checkpoints/{RUN_NAME} \
    --epochs 60 \
    --batch_size 64 \
    --learning_rate 3e-4 \
    --temperature 0.5 \
    --workers 4 \
    --val_frac 0.1 \
    --save_every 10 \
    --seed 42 \
    --tb_log_dir /content/drive/MyDrive/diploma_logs/{RUN_NAME}

In [ ]:
# === 3b: SimCLR — resume после обрыва Colab-сессии ===
# Благодаря полному формату чекпоинта optimizer/scheduler восстанавливаются
# в точности — первые шаги после resume идут без потери momentum.

RUN_NAME = "simclr_v2_t05_default_aug"
LAST_EPOCH = 30           # <-- номер последнего завершённого чекпоинта
TOTAL_EPOCHS = 60
REMAINING = TOTAL_EPOCHS - LAST_EPOCH

!python /content/diploma_project/src/models/deep_clustering/train.py \
    --data_dir /content/diploma_project/data/processed \
    --metadata_path /content/diploma_project/data/processed/tiles_metadata.csv \
    --output_dir /content/drive/MyDrive/diploma_checkpoints/{RUN_NAME} \
    --epochs {REMAINING} \
    --batch_size 64 \
    --workers 4 \
    --val_frac 0.1 \
    --seed 42 \
    --tb_log_dir /content/drive/MyDrive/diploma_logs/{RUN_NAME} \
    --resume /content/drive/MyDrive/diploma_checkpoints/{RUN_NAME}/simclr_resnet50_epoch_{LAST_EPOCH}.pth \
    --start_epoch {LAST_EPOCH}

In [ ]:
# === 3c: SimCLR — альтернативная конфигурация (ablation) ===
# Пример, как менять аугментации и hyperparameters через CLI без правки кода.
# Здесь: более низкий temperature + более агрессивный RandomResizedCrop.

RUN_NAME = "simclr_v2_t02_strong_crop"

!python /content/diploma_project/src/models/deep_clustering/train.py \
    --data_dir /content/diploma_project/data/processed \
    --metadata_path /content/diploma_project/data/processed/tiles_metadata.csv \
    --output_dir /content/drive/MyDrive/diploma_checkpoints/{RUN_NAME} \
    --epochs 60 \
    --batch_size 96 \
    --learning_rate 3e-4 \
    --temperature 0.2 \
    --workers 4 \
    --val_frac 0.1 \
    --save_every 10 \
    --seed 42 \
    --tb_log_dir /content/drive/MyDrive/diploma_logs/{RUN_NAME} \
    --crop_scale_min 0.08 \
    --blur_p 0.5 \
    --noise_std 0.04

In [ ]:
# === 4: BYOL с cosine tau ramp-up ===
# tau растёт от 0.996 до 1.0 по косинусу за весь прогон (Grill et al. 2020).
# best-checkpoint по val_loss благодаря --val_frac.
# 100 эпох: BYOL сходится медленнее SimCLR из-за EMA-обновления target-сети.

RUN_NAME = "byol_cosine_100ep"

!python /content/diploma_project/src/models/deep_clustering/train_byol.py \
    --data_dir /content/diploma_project/data/processed \
    --metadata_path /content/diploma_project/data/processed/tiles_metadata.csv \
    --output_dir /content/drive/MyDrive/diploma_checkpoints/{RUN_NAME} \
    --epochs 100 \
    --batch_size 64 \
    --learning_rate 3e-4 \
    --ema_tau 0.996 \
    --ema_tau_schedule cosine \
    --workers 4 \
    --val_frac 0.1 \
    --save_every 10 \
    --seed 42 \
    --tb_log_dir /content/drive/MyDrive/diploma_logs/{RUN_NAME}

In [ ]:
# === 3d: TensorBoard — live-мониторинг training curves ===
# Посмотреть curves (train/loss_step, val/loss_epoch, grad_norm, tau) прямо в Colab.
# Запускать ПОСЛЕ старта обучения или после нескольких сохранённых логов.

%load_ext tensorboard
%tensorboard --logdir /content/drive/MyDrive/diploma_logs/

In [ ]:
# === 3e: Sweep — серия экспериментов из configs/sweeps/sem_default.yaml ===
# Запускает несколько runs последовательно: SimCLR τ=0.5 (60ep), τ=0.2 (60ep),
# strong_crop, no_blur, BYOL 100ep, BYOL 60ep, BYOL constant_tau.
#
# Используй --only чтобы запустить подмножество, --dry_run чтобы посмотреть команды,
# --skip_extract/--skip_evaluate для разделения этапов.

!python /content/diploma_project/scripts/run_sweep.py \
    --config /content/diploma_project/configs/sweeps/sem_default.yaml \
    --checkpoints_root /content/drive/MyDrive/sweep_ckpts \
    --logs_root /content/drive/MyDrive/sweep_logs \
    --results_root /content/drive/MyDrive/sweep_results \
    --only simclr_t05_default byol_cosine_100ep

---
## 5. Извлечение эмбеддингов

После обучения SimCLR и/или BYOL извлекаем frozen-эмбеддинги для всех тайлов.
Сохраняется в `data/embeddings/{simclr,byol}/finetuned_embeddings.npy` + companion `.csv` с tile_names.

Чекпоинт может быть в любом из форматов (авто-detect):
- новый dict с `model_state_dict`
- legacy raw `state_dict`

Версия projection head (v1/v2) определяется автоматически по BN-ключам.

In [ ]:
# === 4b: Baseline — ImageNet ResNet50 (без дообучения) ===
# Извлекаем эмбеддинги из pretrained ResNet50 для сравнения.
# Сохраняет resnet50_embeddings.npy + embedding_names.csv в data/embeddings/.
# Нужно запустить ОДИН РАЗ перед evaluation.

!python /content/diploma_project/src/models/deep_clustering/extract_simclr_embeddings.py \
    --checkpoint IMAGENET \
    --data_dir /content/diploma_project/data/processed \
    --metadata_path /content/diploma_project/data/processed/tiles_metadata.csv \
    --output_dir /content/diploma_project/data/embeddings \
    --batch_size 128 \
    --workers 4 \
    --seed 42

In [ ]:
# === 5: Извлечение эмбеддингов из best-checkpoint ===
# Поддерживает оба формата чекпоинта: новый dict с model_state_dict
# и legacy raw state_dict (auto-detect в SimCLR.from_state_dict).
# Версия projection head определяется по BN-ключам автоматически.

RUN_NAME = "simclr_v2_t05_default_aug"
CHECKPOINT = f'/content/drive/MyDrive/diploma_checkpoints/{RUN_NAME}/simclr_resnet50_best.pth'
MODEL_TYPE = 'simclr'  # 'simclr' или 'byol'
OUTPUT_SUBDIR = 'simclr'  # эмбеддинги SimCLR пойдут в data/embeddings/simclr/

# Для BYOL:
# RUN_NAME = "byol_cosine_100ep"
# CHECKPOINT = f'/content/drive/MyDrive/diploma_checkpoints/{RUN_NAME}/byol_resnet50_best.pth'
# MODEL_TYPE = 'byol'
# OUTPUT_SUBDIR = 'byol'

# Для SimCLR strong_crop (сильные аугментации, crop_scale_min=0.08 из sem_default.yaml):
# RUN_NAME = "simclr_t05_strong_crop"
# CHECKPOINT = f'/content/drive/MyDrive/diploma_checkpoints/{RUN_NAME}/simclr_resnet50_best.pth'
# MODEL_TYPE = 'simclr'
# OUTPUT_SUBDIR = 'simclr_strong_crop'

!python /content/diploma_project/src/models/deep_clustering/extract_simclr_embeddings.py \
    --checkpoint {CHECKPOINT} \
    --model_type {MODEL_TYPE} \
    --data_dir /content/diploma_project/data/processed \
    --metadata_path /content/diploma_project/data/processed/tiles_metadata.csv \
    --output_dir /content/diploma_project/data/embeddings/{OUTPUT_SUBDIR} \
    --batch_size 128 \
    --workers 4 \
    --seed 42

In [ ]:
# === 5b: Опциональная UMAP-визуализация (baseline vs fine-tuned) ===
# extract_simclr_embeddings.py генерирует PNG как побочный артефакт.
# Этот сам ячейка — просто показать его в Colab.

from IPython.display import Image, display
import os

for subdir in ['simclr', 'byol']:
    umap_path = f'/content/diploma_project/data/embeddings/{subdir}/baseline_vs_{subdir}_umap.png'
    if os.path.exists(umap_path):
        print(f"\nUMAP: Baseline vs {subdir.upper()}")
        display(Image(umap_path))

In [ ]:
# === 6a: Sanity check — все эмбеддинги на месте ===
import os

EMB_DIR = '/content/diploma_project/data/embeddings'

expected = [
    ('Baseline ResNet50',  'resnet50_embeddings.npy',                    'embedding_names.csv'),
    ('SimCLR',             'simclr/finetuned_embeddings.npy',            'simclr/finetuned_embedding_names.csv'),
    ('SimCLR strong_crop', 'simclr_strong_crop/finetuned_embeddings.npy','simclr_strong_crop/finetuned_embedding_names.csv'),
    ('BYOL',               'byol/finetuned_embeddings.npy',              'byol/finetuned_embedding_names.csv'),
]
for name, emb_rel, names_rel in expected:
    emb = os.path.join(EMB_DIR, emb_rel)
    names = os.path.join(EMB_DIR, names_rel)
    emb_ok = os.path.exists(emb)
    names_ok = os.path.exists(names)
    status = "OK" if (emb_ok and names_ok) else "MISSING"
    print(f"  [{status}] {name}: emb={emb_ok}, names={names_ok}")

# Если embedding_names.csv устарел — регенерация в один клик:
# !python /content/diploma_project/scripts/regenerate_embedding_names.py --all

In [ ]:
# === 6b: Unified SEM Evaluation — четыре оценки + markdown-отчёт ===
# Запускает Cross-Scale Retrieval, Scale Invariance, Linear Probe, k-NN Probe.
# Все интервальные метрики снабжены bootstrap 95% CI.
# Итог: data/results/sem_eval_report_<timestamp>.md + отдельные CSV.

%cd /content/diploma_project

!python src/evaluation/run_sem_evaluation.py \
    --meta_path data/processed/tiles_metadata.csv \
    --emb_dir data/embeddings \
    --annotations_path data/sft_annotations.csv \
    --output_dir /content/drive/MyDrive/diploma_results/ \
    --K 10 \
    --n_clusters 4 \
    --n_folds 5 \
    --knn_k 1 5 20 \
    --seed 42

In [ ]:
# === 6c: Показать markdown-отчёт прямо в ноутбуке ===
from IPython.display import display, Markdown
import glob, os

RESULTS = '/content/drive/MyDrive/diploma_results'

reports = sorted(glob.glob(f'{RESULTS}/sem_eval_report_*.md'))
if reports:
    latest = reports[-1]
    print(f"Latest report: {latest}\n")
    with open(latest, 'r', encoding='utf-8') as f:
        display(Markdown(f.read()))
else:
    print(f"No markdown reports in {RESULTS}. Run the evaluation cell above first.")

In [ ]:
# === 6d: Артефакты на Drive — полный список для сдачи ===
import glob, os

def list_dir(path, label):
    print(f"\n{label}  ({path}):")
    for f in sorted(glob.glob(f'{path}/**/*', recursive=True)):
        if os.path.isfile(f):
            size_mb = os.path.getsize(f) / (1024 * 1024)
            rel = os.path.relpath(f, path)
            suffix = f"{size_mb:8.2f} MB" if size_mb >= 0.1 else f"{size_mb*1024:8.1f} KB"
            print(f"  {rel:60s} {suffix}")

list_dir('/content/drive/MyDrive/diploma_results', 'Results (CSV + markdown)')
list_dir('/content/drive/MyDrive/diploma_checkpoints', 'Checkpoints (.pth)')
list_dir('/content/drive/MyDrive/diploma_logs', 'TensorBoard logs')

print("\nDone — all artifacts saved to Google Drive.")

---
## 7. Дополнительные эксперименты

Эксперименты, необходимые для закрытия вопросов из ревью диплома:

- **Exp 1 — Unified N**: пересечение тайлов всех моделей + пересчёт eval на едином наборе
- **Exp 2 — Cramér's V k-sweep**: проверка гипотезы артефакта малого k=4
- **Exp 4 — DINOv2 frozen features**: сравнение с ViT-B/14 без дообучения

### 7a. Experiment 1: Unified N

Проблема: разные модели оценены на разных подмножествах тайлов (N от 5950 до 9898).
Решение: inner join всех `embedding_names`, фильтрация эмбеддингов до пересечения, перезапуск eval.

In [ ]:
# === 7a: Unified N — пересечение тайлов + пересчёт eval ===
# Шаг 1: Находим общие тайлы (inner join по embedding_names всех моделей)
# Шаг 2: Фильтруем эмбеддинги каждой модели до этого пересечения
# Шаг 3: Перезапускаем eval на унифицированном наборе
#
# ~15-30 минут, GPU не нужен.

import os, sys
import numpy as np
import pandas as pd
from pathlib import Path

sys.path.append('/content/diploma_project')
from src.evaluation.eval_utils import MODEL_CONFIGS, MODEL_NAMES_FILES

EMB_DIR = Path('/content/diploma_project/data/embeddings')
UNIFIED_DIR = Path('/content/diploma_project/data/embeddings_unified')

# --- Шаг 1: Inner join ---
sets = []
for model_name, names_file in MODEL_NAMES_FILES.items():
    path = EMB_DIR / names_file
    if path.exists():
        df = pd.read_csv(path)
        col = "tile_name" if "tile_name" in df.columns else df.columns[0]
        s = set(df[col].values)
        sets.append(s)
        print(f"  {model_name}: {len(s)} tiles")

common = sets[0]
for s in sets[1:]:
    common &= s
common_tiles = sorted(common)
print(f"\n  Intersection: {len(common_tiles)} tiles")

# --- Шаг 2: Фильтрация ---
UNIFIED_DIR.mkdir(parents=True, exist_ok=True)
pd.DataFrame({"tile_name": common_tiles}).to_csv(
    UNIFIED_DIR / "unified_embedding_names.csv", index=False)

for model_name in MODEL_CONFIGS:
    emb_path = EMB_DIR / MODEL_CONFIGS[model_name]
    names_path = EMB_DIR / MODEL_NAMES_FILES[model_name]
    if not emb_path.exists():
        print(f"  SKIP {model_name}")
        continue

    embeddings = np.load(emb_path)
    names = pd.read_csv(names_path)
    col = "tile_name" if "tile_name" in names.columns else names.columns[0]
    name_to_idx = {n: i for i, n in enumerate(names[col].values)}
    indices = [name_to_idx[t] for t in common_tiles if t in name_to_idx]

    slug = model_name.replace(" ", "_").replace("(", "").replace(")", "").replace("=", "")
    model_dir = UNIFIED_DIR / slug
    model_dir.mkdir(parents=True, exist_ok=True)
    np.save(model_dir / "finetuned_embeddings.npy", embeddings[indices])
    pd.DataFrame({"tile_name": [common_tiles[i] for i in range(len(common_tiles)) if common_tiles[i] in name_to_idx]}).to_csv(
        model_dir / "finetuned_embedding_names.csv", index=False)
    print(f"  {model_name}: {len(indices)} tiles -> {model_dir}")

print(f"\nUnified embeddings saved to {UNIFIED_DIR}")

In [ ]:
# === 7a (продолжение): Eval на unified эмбеддингах ===
# Перезапускаем полный eval-пайплайн, указывая unified dir.

%cd /content/diploma_project

# Временно подменяем пути в eval_utils
from src.evaluation import eval_utils
import importlib

_orig_configs = eval_utils.MODEL_CONFIGS.copy()
_orig_names = eval_utils.MODEL_NAMES_FILES.copy()
_orig_dir = eval_utils.DEFAULT_EMB_DIR

for model_name in _orig_configs:
    slug = model_name.replace(" ", "_").replace("(", "").replace(")", "").replace("=", "")
    eval_utils.MODEL_CONFIGS[model_name] = f"{slug}/finetuned_embeddings.npy"
    eval_utils.MODEL_NAMES_FILES[model_name] = f"{slug}/finetuned_embedding_names.csv"
eval_utils.DEFAULT_EMB_DIR = UNIFIED_DIR

!python src/evaluation/run_sem_evaluation.py \
    --meta_path data/processed/tiles_metadata.csv \
    --emb_dir {str(UNIFIED_DIR)} \
    --annotations_path data/sft_annotations.csv \
    --output_dir /content/drive/MyDrive/diploma_results/unified_n/ \
    --K 10 \
    --n_clusters 4 \
    --n_folds 5 \
    --knn_k 1 5 20 \
    --seed 42

# Восстанавливаем оригинальные пути
eval_utils.MODEL_CONFIGS.update(_orig_configs)
eval_utils.MODEL_NAMES_FILES.update(_orig_names)
eval_utils.DEFAULT_EMB_DIR = _orig_dir
print("Eval configs restored.")

### 7b. Experiment 2: Cramér's V at k ∈ {4, 8, 15, 20}

Гипотеза: высокий Cramér's V (cluster vs magnification) для контрастивных моделей при k=4 — артефакт малого числа кластеров. При росте k семантические и масштабные кластеры должны разделиться, и V должен снижаться.

~5 минут, GPU не нужен.

In [ ]:
# === 7b: Cramér's V k-sweep ===
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics import adjusted_mutual_info_score
from scipy.stats import chi2_contingency

EMB_DIR = Path('/content/diploma_project/data/embeddings')
META_PATH = Path('/content/diploma_project/data/processed/tiles_metadata.csv')
K_VALUES = [4, 8, 15, 20]
SEED = 42

from src.evaluation.eval_utils import MODEL_CONFIGS, MODEL_NAMES_FILES

def cramers_v(x, y):
    ct = pd.crosstab(pd.Series(x), pd.Series(y))
    if ct.shape[0] < 2 or ct.shape[1] < 2:
        return 0.0
    chi2, _, _, _ = chi2_contingency(ct)
    n = ct.values.sum()
    return float(np.sqrt(chi2 / (n * (min(ct.shape) - 1))))

def mag_var_ratio(labels, magnifications):
    log_mag = np.log10(magnifications.astype(float) + 1)
    total_var = np.var(log_mag)
    if total_var == 0:
        return 0.0
    within_var = sum(m.sum() * np.var(log_mag[m]) for c in np.unique(labels)
                     if (m := labels == c).sum() > 1) / len(log_mag)
    return float(within_var / total_var)

meta = pd.read_csv(META_PATH)
results = []

for model_name in MODEL_CONFIGS:
    emb_path = EMB_DIR / MODEL_CONFIGS[model_name]
    names_path = EMB_DIR / MODEL_NAMES_FILES[model_name]
    if not emb_path.exists():
        continue

    embeddings = np.load(emb_path)
    names_df = pd.read_csv(names_path)
    col = "tile_name" if "tile_name" in names_df.columns else names_df.columns[0]
    tile_names = names_df[col].values

    meta_aligned = meta.set_index("tile_name").loc[tile_names].reset_index()
    magnifications = meta_aligned["magnification"].values

    norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
    norms[norms == 0] = 1
    emb_norm = embeddings / norms

    print(f"\n{model_name} ({len(embeddings)} tiles)")
    for k in K_VALUES:
        labels = MiniBatchKMeans(n_clusters=k, random_state=SEED, batch_size=1024).fit_predict(emb_norm)
        v = cramers_v(labels, magnifications)
        ami = adjusted_mutual_info_score(labels, magnifications)
        mvr = mag_var_ratio(labels, magnifications)
        results.append({"model": model_name, "k": k, "cramers_v": round(v, 4),
                        "ami": round(ami, 4), "mvr": round(mvr, 4)})
        print(f"  k={k:3d}  V={v:.4f}  AMI={ami:.4f}  MVR={mvr:.4f}")

df = pd.DataFrame(results)

# Сохранение
output_path = Path('/content/drive/MyDrive/diploma_results/exp2_cramers_v_ksweep.csv')
output_path.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(output_path, index=False)

# Pivot для наглядности
print("\n" + "="*60)
print("Cramér's V pivot (model × k):")
print("="*60)
print(df.pivot(index="model", columns="k", values="cramers_v").to_string())
print(f"\nSaved to {output_path}")

### 7c. Experiment 4: DINOv2 ViT-B/14 Frozen Features

Извлекаем 768-мерные эмбеддинги из DINOv2 ViT-B/14 (frozen, без дообучения) для всех тайлов,
затем запускаем тот же eval-пайплайн. Это даёт сравнение с современным ViT-based feature extractor.

~1-2 часа, **нужен GPU** (T4 достаточно).

In [ ]:
# === 7c: DINOv2 — извлечение эмбеддингов ===
import torch
import numpy as np
import pandas as pd
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from tqdm import tqdm

TILES_DIR = Path('/content/diploma_project/data/processed')
EMB_DIR = Path('/content/diploma_project/data/embeddings')
DINOV2_DIR = EMB_DIR / 'dinov2'
BATCH_SIZE = 64
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# Берём список тайлов из Baseline (для выравнивания)
from src.evaluation.eval_utils import MODEL_NAMES_FILES
names_df = pd.read_csv(EMB_DIR / MODEL_NAMES_FILES["Baseline (ImageNet)"])
col = "tile_name" if "tile_name" in names_df.columns else names_df.columns[0]
tile_names = names_df[col].tolist()
print(f"Tiles to process: {len(tile_names)}")

# Dataset
class TileDataset(Dataset):
    def __init__(self, names, tiles_dir, transform):
        self.names = names
        self.tiles_dir = tiles_dir
        self.transform = transform

    def __len__(self):
        return len(self.names)

    def __getitem__(self, idx):
        name = self.names[idx]
        for ext in [".png", ".jpg", ".jpeg", ".tiff", ".tif", ""]:
            path = self.tiles_dir / f"{name}{ext}" if ext else self.tiles_dir / name
            if path.exists():
                img = Image.open(path).convert("RGB")
                return self.transform(img), name
        raise FileNotFoundError(f"Tile not found: {name}")

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Load DINOv2
print(f"Loading DINOv2 ViT-B/14 on {DEVICE}...")
model = torch.hub.load("facebookresearch/dinov2", "dinov2_vitb14")
model = model.to(DEVICE).eval()

dataset = TileDataset(tile_names, TILES_DIR, transform)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2,
                    pin_memory=(DEVICE == 'cuda'))

all_emb, all_names = [], []
with torch.no_grad():
    for images, names in tqdm(loader, desc="DINOv2 extraction"):
        features = model(images.to(DEVICE))  # (B, 768)
        all_emb.append(features.cpu().numpy())
        all_names.extend(names)

embeddings = np.concatenate(all_emb, axis=0)
print(f"Extracted: {embeddings.shape}")

# Save
DINOV2_DIR.mkdir(parents=True, exist_ok=True)
np.save(DINOV2_DIR / "finetuned_embeddings.npy", embeddings)
pd.DataFrame({"tile_name": all_names}).to_csv(
    DINOV2_DIR / "finetuned_embedding_names.csv", index=False)
print(f"Saved to {DINOV2_DIR}")

In [ ]:
# === 7c (продолжение): Eval с DINOv2 в сравнении ===
# Добавляем DINOv2 в MODEL_CONFIGS и перезапускаем eval.

from src.evaluation import eval_utils

# Добавляем DINOv2 к списку моделей
eval_utils.MODEL_CONFIGS["DINOv2 ViT-B/14"] = "dinov2/finetuned_embeddings.npy"
eval_utils.MODEL_NAMES_FILES["DINOv2 ViT-B/14"] = "dinov2/finetuned_embedding_names.csv"

!python src/evaluation/run_sem_evaluation.py \
    --meta_path data/processed/tiles_metadata.csv \
    --emb_dir data/embeddings \
    --annotations_path data/sft_annotations.csv \
    --output_dir /content/drive/MyDrive/diploma_results/with_dinov2/ \
    --K 10 \
    --n_clusters 4 \
    --n_folds 5 \
    --knn_k 1 5 20 \
    --seed 42

# Убираем DINOv2 из глобального конфига после eval
del eval_utils.MODEL_CONFIGS["DINOv2 ViT-B/14"]
del eval_utils.MODEL_NAMES_FILES["DINOv2 ViT-B/14"]
print("Done. DINOv2 removed from global configs.")